# 9.1 Rechteck- und Trapezregel

In Kapitel 7 haben wir aus diskreten Positionsmessungen eine Geschwindigkeit
berechnet: Ableitung als Differenzenquotient. In Kapitel 8 haben wir eine
Kostenfunktion minimiert, um Modellparameter aus Messdaten zu schätzen. Jetzt
kehren wir zur Kinematik zurück und stellen die Frage anders: Ein Sensor
liefert den Geschwindigkeitsverlauf $v(t_i)$ eines bremsenden Fahrzeugs. Wie
weit fährt es noch, bis es steht?

Der Bremsweg ist das Integral $s = \int_0^T v(t)\,\mathrm{d}t$. Die
Geschwindigkeit liegt aber nur als Zahlenfolge vor, nicht als Formel. Wir
brauchen ein Verfahren, das ein bestimmtes Integral aus diskreten Messwerten
berechnet. Das nennt man **numerische Quadratur**.

## Lernziele

* [ ] Sie können die **linke und rechte Rechtecksregel** (oft auch *Linkssumme*
  bzw. *Rechtssumme* genannt) und die **Mittelpunktregel** als Varianten der
  Rechteckregel erklären und implementieren.
* [ ] Sie können begründen, welche Variante bei einem monoton fallenden
  Geschwindigkeitsverlauf über- und welche unterschätzt, ohne Code
  auszuführen.
* [ ] Sie können die **Trapezregel** aus der Idee der linearen Interpolation
  herleiten und als Python-Schleife implementieren.
* [ ] Sie können `np.trapezoid` einsetzen und das Ergebnis mit der eigenen
  Implementierung vergleichen.

## Die Rechteckregel

Wir erzeugen zunächst ein vereinfachtes Bremsmodell: das Fahrzeug startet bei
100 km/h und bremst gleichmäßig (mit konstanter Verzögerung, in der Realität
nur näherungsweise erfüllt) bis zum Stillstand in 5 Sekunden. Physikalisch
entspricht gleichmäßig bremsen einer **konstanten Verzögerung** $a$. Für
konstante $a$ gilt $v(t) = v_0 + a t$; mit der Bedingung $v(T) = 0$ folgt $a =
-v_0/T$ und damit

\begin{equation*}
v(t) = v_0 \bigl(1 - t/T\bigr).
\end{equation*}

Diesen linearen Verlauf setzen wir im Code bei den Messzeitpunkten $t_i$ als
$v(t_i)$ ein. Der exakte Bremsweg ergibt sich aus dem bestimmten Integral $s =
\int_0^T v_0(1 - t/T)\,\mathrm{d}t$ bzw. $s = v_0 \cdot \tfrac{T}{2}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.style as style
style.use('seaborn-v0_8')

# --- Bremsvorgang: diskrete Messwerte ---
# Anfangsgeschwindigkeit 100 km/h = 27.78 m/s, lineare Verzögerung bis v = 0
v0 = 100 / 3.6  # Anfangsgeschwindigkeit in m/s
T  = 5.0        # Bremsdauer in s
n  = 6          # Anzahl Messpunkte (bewusst klein für Visualisierung)

t_mess = np.linspace(0, T, n)   # Zeitpunkte in s
v_mess = v0 * (1 - t_mess / T)  # Geschwindigkeit in m/s (linear fallend)

# Analytischer Bremsweg als Referenz: Integral von v0*(1 - t/T) von 0 bis T
bremsweg_exakt = v0 * T / 2     # = 69.44 m

print(f"Anfangsgeschwindigkeit: {v0:.2f} m/s ({v0 * 3.6:.1f} km/h)")
print(f"Bremsdauer:             {T:.1f} s")
print(f"Exakter Bremsweg:       {bremsweg_exakt:.2f} m")
print()
print("Messwerte:")
for ti, vi in zip(t_mess, v_mess):
    print(f"  t = {ti:.1f} s  -->  v = {vi:.2f} m/s")

# Visualisierung
fig, ax = plt.subplots()
ax.scatter(t_mess, v_mess, color='#E60000', s=40)
ax.set_title('Bremsvorgang mit konstanter Verzögerung -5.55 m/s^2')
ax.set_xlabel('Zeit in s')
ax.set_ylabel('Geschwindigkeit in m/s')
ax.grid(True)


Mit $n = 6$ Messpunkten entsteht ein Zeitraster mit Schrittweite
$\Delta t = 1$ s. Die einfachste Näherung für das Integral ersetzt die
Fläche unter der Kurve durch Rechtecke. Je nachdem, welcher Funktionswert
die Höhe des Rechtecks bestimmt, unterscheiden wir drei Varianten.

In [ ]:
import numpy as np

# --- Schrittweite (gleichmäßiges Raster vorausgesetzt) ---
dt = t_mess[1] - t_mess[0]   # Schrittweite in s

# --- Linkssumme: Höhe = v am linken Rand jedes Intervalls ---
# Intervall [t_0, t_1], [t_1, t_2], ..., [t_{n-2}, t_{n-1}]
# verwendet v_mess[0], v_mess[1], ..., v_mess[n-2]  (alle außer dem letzten)
s_links = np.sum(v_mess[:-1]) * dt

# --- Rechtssumme: Höhe = v am rechten Rand jedes Intervalls ---
# verwendet v_mess[1], v_mess[2], ..., v_mess[n-1]  (alle außer dem ersten)
s_rechts = np.sum(v_mess[1:]) * dt

# --- Mittelpunktregel: Höhe = v in der Mitte jedes Intervalls ---
# Mittelpunkte durch Mittelwert benachbarter Messpunkte
t_mitte = 0.5 * (t_mess[:-1] + t_mess[1:])
v_mitte = 0.5 * (v_mess[:-1] + v_mess[1:])
s_mitte = np.sum(v_mitte) * dt

print(f"Exakter Bremsweg:  {bremsweg_exakt:.4f} m")
print()
print(f"Linkssumme:        {s_links:.4f} m  (Fehler: {s_links - bremsweg_exakt:+.4f} m)")
print(f"Rechtssumme:       {s_rechts:.4f} m  (Fehler: {s_rechts - bremsweg_exakt:+.4f} m)")
print(f"Mittelpunktregel:  {s_mitte:.4f} m  (Fehler: {s_mitte - bremsweg_exakt:+.4f} m)")

Die Ergebnisse unterscheiden sich deutlich. Um zu verstehen warum, visualisieren
wir alle drei Varianten.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.style as style
style.use('seaborn-v0_8')

# --- Feiner aufgelöste Kurve als Referenz ---
t_fein = np.linspace(0, T, 300)
v_fein = v0 * (1 - t_fein / T)

fig, ax = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
varianten = [
    ("Linkssumme",       v_mess[:-1], t_mess[:-1]),
    ("Rechtssumme",      v_mess[1:],  t_mess[:-1]),
    ("Mittelpunktregel", v_mitte,     t_mess[:-1]),
]

for axes, (titel, hoehen, x_start) in zip(ax, varianten):
    # Rechtecke einzeichnen
    for xi, hi in zip(x_start, hoehen):
        axes.bar(xi, hi, width=dt, align='edge',
                 color='#CCDEE9', edgecolor='#005A94', linewidth=1.2)
    # Genaue Kurve darüber
    axes.plot(t_fein, v_fein, color='#005A94', linewidth=2, label='v(t) exakt')
    axes.scatter(t_mess, v_mess, color='#E60000', s=40)
    axes.set_title(titel)
    axes.set_xlabel('Zeit in s')
    axes.grid(True)

ax[0].set_ylabel('Geschwindigkeit in m/s')
plt.tight_layout()
plt.show()

### Mini-Übung 1

Betrachten Sie die drei Plots für den monoton fallenden Geschwindigkeitsverlauf.

1. Welche Variante überschätzt den Bremsweg, welche unterschätzt ihn?
   Begründen Sie Ihre Antwort in einem Satz, ohne Code auszuführen.
2. Wie würde sich die Antwort ändern, wenn das Fahrzeug nicht bremsen,
   sondern beschleunigen würde (monoton steigender Geschwindigkeitsverlauf)?
3. Die Mittelpunktregel hat hier keinen Fehler. Liegt das an der Methode oder
   an der Geschwindigkeitsfunktion? Begründen Sie kurz.

## Die Trapezregel

Die Rechteckregel approximiert die Fläche unter der Kurve durch
Rechtecke. Eine bessere Idee: wir verbinden benachbarte Messpunkte durch
Geraden und berechnen die Fläche unter diesem Streckenzug. Jedes Teilstück
ergibt ein **Trapez**.

Für ein einzelnes Intervall $[t_i, t_{i+1}]$ ist die Trapezfläche der
Mittelwert der beiden Randwerte mal die Breite:

$$\Delta s_i = \frac{v(t_i) + v(t_{i+1})}{2} \cdot \Delta t.$$

Die Summe über alle Intervalle ergibt die **Trapezregel**:

\begin{equation*}
s \approx \Delta t \cdot \left(\frac{v(t_0)}{2} + v(t_1) + \cdots +
v(t_{n-2}) + \frac{v(t_{n-1})}{2}\right).
\end{equation*}

Die Randpunkte gehen also mit dem halben Gewicht ein, alle inneren Punkte
mit vollem Gewicht.

In [ ]:
import numpy as np

# --- Trapezregel: manuelle Implementierung ---
# Wir iterieren über alle n-1 Intervalle und addieren die Trapezflächen.
dt = t_mess[1] - t_mess[0]

bremsweg_trapez = 0.0
for i in range(len(v_mess) - 1):
    # Trapezfläche = Mittelwert der beiden Randwerte * Breite
    trapez = 0.5 * (v_mess[i] + v_mess[i + 1]) * dt
    bremsweg_trapez += trapez

print(f"Trapezregel (Schleife): {bremsweg_trapez:.4f} m")
print(f"Exakter Bremsweg:       {bremsweg_exakt:.4f} m")
print(f"Fehler:                 {bremsweg_trapez - bremsweg_exakt:+.4f} m")

Der folgende Python-Code visualisert die Trapezregel.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.style as style
style.use('seaborn-v0_8')

# --- Feiner aufgelöste Kurve als Referenz ---
t_fein = np.linspace(0, T, 300)
v_fein = v0 * (1 - t_fein / T)

fig, ax = plt.subplots(figsize=(6, 4))

# --- Trapeze einzeichnen: lineare Verbindung der Messpunkte ---
for i in range(len(t_mess) - 1):
    x_trap = [t_mess[i],   t_mess[i + 1]]
    y_trap = [v_mess[i],   v_mess[i + 1]]
    ax.fill_between(x_trap, y_trap, color='#CCDEE9',
                    edgecolor='#005A94', linewidth=1.2, alpha=0.9)

# Exakte Kurve und Messpunkte
ax.plot(t_fein, v_fein, color='#005A94', linewidth=2, label='v(t) exakt')
ax.scatter(t_mess, v_mess, color='#E60000', s=40, label='Messpunkte')

ax.set_title('Trapezregel für den Bremsvorgang')
ax.set_xlabel('Zeit in s')
ax.set_ylabel('Geschwindigkeit in m/s')
ax.grid(True)
ax.legend()
plt.tight_layout()
plt.show()

NumPy bietet seit Version 2.0 die Funktion `np.trapezoid`, die dieselbe Rechnung
in einer Zeile erledigt. In älteren NumPy-Versionen heißt die Funktion
`np.trapz`; sie ist inhaltlich gleich.

In [ ]:
import numpy as np

# --- Trapezregel: np.trapezoid (seit NumPy 2.0) ---
# np.trapezoid(y, x) berechnet das Integral der Werte y über die Stützstellen x.
# Das x-Array kann ungleichmäßige Abstände haben; hier ist es gleichmäßig.
bremsweg_np = np.trapezoid(v_mess, t_mess)

print(f"np.trapezoid:     {bremsweg_np:.4f} m")
print(f"Schleife:         {bremsweg_trapez:.4f} m")
print(f"Übereinstimmung:  {np.isclose(bremsweg_np, bremsweg_trapez)}")

Die Trapezregel liefert für diesen linearen Geschwindigkeitsverlauf das
exakte Ergebnis: Die lineare Verbindung zweier Punkte auf einer Geraden
liegt exakt auf der Geraden. Für nichtlineare Verläufe entsteht ein Fehler,
dessen Größe wir in Abschnitt 9.3 analysieren.

### Mini-Übung 2

Ein Fahrzeug misst die Geschwindigkeit an drei Zeitpunkten:

| $t$ in s | 0 | 1 | 2 |
|----------|---|---|---|
| $v$ in m/s | 20 | 15 | 5 |

1. Berechnen Sie den Bremsweg mit der Trapezregel per Handrechnung.
   Zeigen Sie beide Trapezflächen einzeln.
2. Implementieren Sie die Berechnung mit `np.trapezoid` und überprüfen
   Sie Ihr Handrechenresultat.
3. Warum ergibt die Linkssumme für diese drei Punkte einen anderen Wert?
   Berechnen Sie auch diesen per Hand und vergleichen Sie.

In [ ]:
# Code-Zelle

## Zusammenfassung und Ausblick

Die Rechteckregel nähert das Integral durch konstante Funktionswerte
pro Intervall an. Linkssumme und Rechtssumme liefern bei monotonen
Verläufen einen systematischen Fehler, der sich erst mit vielen
Messpunkten verringert. Die Trapezregel verbessert die Näherung durch
lineare Interpolation zwischen den Messpunkten. Mit `np.trapezoid` steht
sie als fertiges NumPy-Werkzeug zur Verfügung.

In Abschnitt 9.2 wenden wir die Trapezregel auf einen realistischen
Notbremsvorgang an. In Abschnitt 9.3 lernen wir die Simpson-Regel kennen,
die durch quadratische Interpolation eine wesentlich höhere Genauigkeit
erreicht.